# Edit Wikidata using wikibaseintegrator

In [ ]:
from wikibaseintegrator import WikibaseIntegrator
from wikibaseintegrator import wbi_login
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator import datatypes
from wikibaseintegrator.wbi_enums import ActionIfExists
import logging
import json
import random
from pathlib import Path

with Path('../authorization.json').open(mode='r') as authorization_file:
    authorization = json.load(authorization_file)

USER_AUTH = authorization['user_auth']
USERNAME = authorization['username']
PASSWORD = authorization['password']
CONSUMER_TOKEN = authorization['consumer_token']
CONSUMER_SECRET = authorization['consumer_secret']

BOTNAME = authorization['botname']
    

logging.basicConfig(filename='wikibaseint-debug.log', 
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)


wbi_config['USER_AGENT'] = f'{BOTNAME} (https://www.wikidata.org/wiki/User:{USERNAME})'
wbi_config['MEDIAWIKI_API_URL'] = 'https://test.wikidata.org/w/api.php'

PROPS = {'instance_of':'P31',
        'author':'P50',
         'title':'P1476',
         'has_edition':'P747',
         'language':'P407',
         'publication_date':'P577'}
ENTITIES = {
   'literary_work':'Q7725634', 
   'German':'Q188',
   'Hugo_Ball':'Q70989'
}

### Logging in using the old method

In [ ]:
login_instance = wbi_login.Login(user=USER_AUTH, password=PASSWORD)

wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'

### Logging in using OAuth (preferred)

In [ ]:
oauth = wbi_login.OAuth2(consumer_token=CONSUMER_TOKEN, 
                                  consumer_secret=CONSUMER_SECRET)
wbi_oauth = WikibaseIntegrator(login=oauth)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'


Get this entity to test:  https://www.wikidata.org/wiki/Q105624761

## Add a claim

What titles there are now?

In [ ]:
flametti = wbi.item.get(entity_id='Q105624761')

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

Now we add this title to our local python representation of this entity

In [ ]:
title_en_string = datatypes.MonolingualText(text='Flametti, or The Dandyism of the Poor', language='en', prop_nr=PROPS['title'])
title_en_string

flametti.claims.add(title_en_string)

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

## Write it to the instance

In [ ]:
flametti = flametti.write(login=login_instance)

[claim['mainsnak']['datavalue']['value'] for claim in flametti.get_json()['claims'][PROPS['title']]]

See:  https://www.wikidata.org/wiki/Q105624761

### With previous versions of WikibaseIntegrator, one had to do it like this:

In [ ]:
flametti.claims.add(title_en_string, action_if_exists=ActionIfExists.KEEP)